# Multi-Dataset SFT & DPO Pipeline

**Generate Training Data from ALL Available Sources**

Combines:
- ✅ Asas Al-Balagha (394 chunks, classical Arabic)
- ✅ Najdi Popular (406 chunks, modern Saudi dialect)

Output:
- Unified SFT dataset with diverse dialects
- Comprehensive DPO pairs
- Proper train/val/test splits possible

## 1. Setup & Environment

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
from getpass import getpass
from collections import defaultdict

# Initialize project path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
    if not (PROJECT_ROOT / 'src').exists():
        raise FileNotFoundError('Could not find src/ directory')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from sft import (
    SFTGenerator,
    SFTValidator,
    QualityScorer,
    load_chunks,
    save_samples,
    DPOPrompts,
)
from sft.generator import extract_answer_without_scaffold

print('✓ All modules imported')
print(f'✓ Project root: {PROJECT_ROOT.name}')

In [ ]:
# Configure environmentdef load_dotenv(path):    if not path.exists():        return    for raw_line in path.read_text(encoding='utf-8').splitlines():        line = raw_line.strip()        if not line or line.startswith('#') or '=' not in line:            continue        name, value = line.split('=', 1)        name, value = name.strip(), value.strip().strip('"').strip("'")        if name and value:            os.environ.setdefault(name, value)load_dotenv(PROJECT_ROOT / '.env')OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY')if not OPENROUTER_API_KEY:    OPENROUTER_API_KEY = getpass('OpenRouter API key: ').strip()    if OPENROUTER_API_KEY:        os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEYMODEL = os.environ.get('OPENROUTER_MODEL', 'google/gemini-2.5-flash-lite')print(f'✓ API configured')print(f'✓ Model: {MODEL}')

## 2. Define All Datasets

In [ ]:
# Define all available datasets
DATASETS = {
    'asas_albalagha': {
        'path': PROJECT_ROOT / 'data' / 'processed' / 'chunks_asas_albalagha.jsonl',
        'name': 'Asas Al-Balagha',
        'dialect': 'Classical Arabic',
        'description': 'Classical Arabic lexicon with diverse meanings',
    },
    'najdi_popular': {
        'path': PROJECT_ROOT / 'data' / 'processed' / 'chunks_najdi_popular.jsonl',
        'name': 'Najdi Popular',
        'dialect': 'Saudi Najdi Dialect',
        'description': 'Modern Najdi dialect popular words and usage',
    },
}

# Output configuration
GENERATED_DIR = PROJECT_ROOT / 'data' / 'generated'
MULTI_DATASET = {
    'sft_candidates': GENERATED_DIR / 'sft' / 'candidates_multi.jsonl',
    'sft_accepted': GENERATED_DIR / 'sft' / 'accepted_multi.jsonl',
    'dpo_candidates': GENERATED_DIR / 'dpo' / 'candidates_multi.jsonl',
}

# Generation settings
SETTINGS = {
    'dpo_target': 5000,
    'max_examples_per_chunk': 12,
    'api_delay': 0.5,
}

print('\n=== DATASETS ===')
for key, config in DATASETS.items():
    exists = '✓' if config['path'].exists() else '✗'
    print(f'{exists} {config["name"]} ({config["dialect"]})')
    print(f'  {config["path"]}')
    if config['path'].exists():
        count = sum(1 for _ in open(config['path']))
        print(f'  {count} chunks')

## 3. Load All Chunks

In [ ]:
print('\n=== LOADING ALL DATASETS ===')

all_chunks = []
dataset_info = {}

for key, config in DATASETS.items():
    if not config['path'].exists():
        print(f'⚠️  {config["name"]}: File not found')
        continue
    
    try:
        chunks = load_chunks(str(config['path']))
        all_chunks.extend(chunks)
        dataset_info[key] = {
            'count': len(chunks),
            'name': config['name'],
            'dialect': config['dialect'],
        }
        print(f'✓ {config["name"]}: {len(chunks)} chunks loaded')
    except Exception as e:
        print(f'✗ {config["name"]}: Error - {e}')

print(f'\n✓ Total chunks: {len(all_chunks):,}')
print(f'  Dataset diversity: {len(dataset_info)} sources')

## 4. Validate Data Integrity

In [ ]:
print('\n=== DATA VALIDATION ===')

# Check for duplicates across datasets
chunk_ids = [c.get('chunk_id') for c in all_chunks]
unique_ids = set(chunk_ids)
duplicates = len(chunk_ids) - len(unique_ids)

if duplicates > 0:
    print(f'⚠️  Found {duplicates} duplicate chunk IDs')
else:
    print(f'✓ All chunk IDs unique')

# Check for completeness
required_fields = ['chunk_id', 'chunk_text']
incomplete = 0
for chunk in all_chunks:
    if not all(chunk.get(f) for f in required_fields):
        incomplete += 1

if incomplete > 0:
    print(f'⚠️  {incomplete} incomplete chunks')
else:
    print(f'✓ All chunks complete')

print(f'\n✓ Data validation passed')

## 5. Generate SFT Samples from All Datasets

In [ ]:
if not OPENROUTER_API_KEY:
    print('⚠️  Generation paused: no API key')
    sft_samples = []
else:
    print(f'\n=== GENERATING SFT FROM {len(dataset_info)} DATASETS ===')
    print(f'Processing {len(all_chunks):,} total chunks\n')
    
    generator = SFTGenerator(OPENROUTER_API_KEY, MODEL)
    
    def progress(curr, total, chunk_id, status):
        pct = (curr / total * 100) if total else 0
        print(f'[{curr:5d}/{total:5d} {pct:5.1f}%] {chunk_id:35s} {status}')
    
    try:
        sft_samples = generator.generate_batch(
            all_chunks,
            limit=None,  # Process all chunks
            max_examples_per_chunk=SETTINGS['max_examples_per_chunk'],
            pause_seconds=SETTINGS['api_delay'],
            progress_callback=progress,
            continue_on_error=True,
        )
        print(f'\n✓ Generated {len(sft_samples):,} SFT samples from all datasets')
    except Exception as e:
        print(f'✗ Generation failed: {e}')
        sft_samples = []

## 6. Filter Quality & Accept

In [ ]:
if sft_samples:
    print('\n=== QUALITY FILTERING ===')
    
    accepted = []
    for sample in sft_samples:
        try:
            scores = QualityScorer.score_sample(sample)
            conf = scores.get('teacher_confidence', 0.0)
            if conf >= 0.7:
                accepted.append(sample)
        except Exception as e:
            continue
    
    rate = len(accepted) / len(sft_samples) * 100 if sft_samples else 0
    print(f'✓ Accepted {len(accepted):,} samples ({rate:.1f}%)')
    
    if accepted:
        try:
            MULTI_DATASET['sft_accepted'].parent.mkdir(parents=True, exist_ok=True)
            from sft import save_samples
            save_samples(accepted, str(MULTI_DATASET['sft_accepted']))
            print(f'✓ Saved to: {MULTI_DATASET["sft_accepted"]}')
        except Exception as e:
            print(f'✗ Error saving: {e}')
else:
    accepted = []
    print('⚠️  No samples to filter')

## 7. Generate DPO Pairs (Scaffold-Free)

In [ ]:
DPO_TYPES = (
    'partial_factual_errors',
    'less_faithful_reconstruction',
    'unsupported_additions',
    'missing_information',
    'wrong_register',
    'weak_organization',
    'poor_instruction_following',
    'wrong_formatting',
    'verbosity',
)

if not OPENROUTER_API_KEY:
    print('⚠️  DPO paused: no API key')
    dpo_candidates = []
elif not accepted:
    print('⚠️  DPO paused: no accepted samples')
    dpo_candidates = []
else:
    print(f'\n=== GENERATING {SETTINGS["dpo_target"]:,}+ DPO PAIRS (SCAFFOLD-FREE) ===')
    print(f'From {len(accepted):,} multi-dataset samples\n')
    
    from sft.schema import SFTSample
    
    generator = SFTGenerator(OPENROUTER_API_KEY, MODEL)
    dpo_candidates = []
    pair_counter = 0
    
    pairs_per_sample = max(1, (SETTINGS['dpo_target'] + len(accepted) - 1) // len(accepted))
    print(f'Strategy: ~{pairs_per_sample} pair(s) per sample\n')
    
    for idx, sft in enumerate(accepted, 1):
        if pair_counter >= SETTINGS['dpo_target']:
            print(f'\n✓ Reached target: {pair_counter:,} DPO pairs')
            break
        
        try:
            inst = sft.messages[0].content
            full_response = sft.messages[1].content
            clean_chosen = extract_answer_without_scaffold(full_response)
            
            for rej_idx, rej_type in enumerate(DPO_TYPES[:pairs_per_sample]):
                if pair_counter >= SETTINGS['dpo_target']:
                    break
                
                try:
                    msgs = DPOPrompts.build_messages(inst, clean_chosen, [rej_type])
                    content, _ = generator.client.call(msgs)
                    
                    resp = json.loads(content)
                    rej = resp.get('rejected', '').strip()
                    
                    if rej:
                        dpo_candidates.append({
                            'pair_id': f'{sft.sample_id}_dpo_{rej_idx:02d}',
                            'source_sample_id': sft.sample_id,
                            'source_chunk_id': sft.chunk_id,
                            'prompt': [{'role': 'user', 'content': inst}],
                            'chosen': [{'role': 'assistant', 'content': clean_chosen}],
                            'rejected': [{'role': 'assistant', 'content': rej}],
                            'rejection_type': rej_type,
                            'verification_status': 'Unverified',
                        })
                        pair_counter += 1
                except:
                    continue
                
                time.sleep(SETTINGS['api_delay'])
        except:
            continue
        
        if idx % 100 == 0:
            print(f'  [{idx:5d}/{len(accepted):5d}] Generated {pair_counter:,} pairs')
    
    print(f'\n✓ Generated {len(dpo_candidates):,} DPO pairs')

## 8. Save All Outputs

In [ ]:
print('\n=== SAVING MULTI-DATASET OUTPUTS ===')

# Save SFT candidates
if sft_samples:
    try:
        MULTI_DATASET['sft_candidates'].parent.mkdir(parents=True, exist_ok=True)
        from sft import save_samples
        save_samples(sft_samples, str(MULTI_DATASET['sft_candidates']))
        print(f'✓ SFT candidates: {len(sft_samples):,} samples')
        print(f'  {MULTI_DATASET["sft_candidates"]}')
    except Exception as e:
        print(f'✗ Error saving SFT: {e}')

# Save DPO candidates
if dpo_candidates:
    try:
        MULTI_DATASET['dpo_candidates'].parent.mkdir(parents=True, exist_ok=True)
        with open(MULTI_DATASET['dpo_candidates'], 'w', encoding='utf-8') as f:
            for pair in dpo_candidates:
                f.write(json.dumps(pair, ensure_ascii=False) + '\n')
        print(f'✓ DPO pairs: {len(dpo_candidates):,} pairs')
        print(f'  {MULTI_DATASET["dpo_candidates"]}')
    except Exception as e:
        print(f'✗ Error saving DPO: {e}')

## 9. Final Summary & Statistics

In [ ]:
print('\n' + '='*70)
print('MULTI-DATASET PIPELINE COMPLETE')
print('='*70)

print('\n📊 SOURCE DATASETS:')
total_chunks = 0
for key, info in dataset_info.items():
    print(f'  ✓ {info["name"]:30s} {info["count"]:5d} chunks ({info["dialect"]})')
    total_chunks += info['count']

print(f'  ─' * 35)
print(f'  Total chunks: {total_chunks:,}')

print('\n📈 GENERATED TRAINING DATA:')
if sft_samples:
    print(f'  SFT Candidates:  {len(sft_samples):>10,} samples')
    print(f'  SFT Accepted:    {len(accepted):>10,} samples ({len(accepted)/len(sft_samples)*100:.1f}%)')
if dpo_candidates:
    print(f'  DPO Pairs:       {len(dpo_candidates):>10,} pairs')
    if accepted:
        ratio = len(dpo_candidates) / len(accepted)
        print(f'  Pairs per SFT:   {ratio:>10.2f}x')

print('\n✅ DATASET DIVERSITY ACHIEVED:')
print(f'  • Classical Arabic (Asas Al-Balagha)')
print(f'  • Modern Saudi Dialect (Najdi Popular)')
print(f'  • Proper document-level train/val/test splits now possible')

print('\n' + '='*70)
print('✓ Ready for training with diverse Arabic data!')
print('='*70)